# QpiAI Quantum SDK extras for the BB84 simulator

Optional. Everything here runs **without** a QpiAI account: if `qpiai-quantum` is not installed or no API key is found, each step logs why and falls back to NumPy.

To use QpiAI: `pip install -r qpiai_extras/requirements.txt`, then either set `QPIAI_API_KEY` in your environment or create `qcloud.env` in the repository root with `API_KEY="..."`. `qcloud.env` is git-ignored. **Never commit a key.**

Note from the SDK documentation: on the local simulator `QpiAI-QSV-Local` the QRNG samples with NumPy, so it is pseudo-random. Only a QPU device gives hardware quantum randomness.

In [ ]:
import sys, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'qpiai_extras' else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO)
from qpiai_extras import qpiai_status, QRNGSource
print(qpiai_status(login=False))

## (a) QRNG bits and bases for Alice
`QRNGSource` behaves like a NumPy generator for `integers(0, 2, size)`, so it can replace Alice's random source in the pulse-level simulator.

In [ ]:
from src.simulator.bb84_engine import BB84Simulator
q = QRNGSource(fallback_seed=0)
print(q.source_label)
sim = BB84Simulator(raw_photons_per_block=4096, random_state=3)
sim.alice_source = q
seq = sim.simulate_session(1, 1, attack_mode='multiattack', blocks_per_session=60, attack_start_block=31, p_eve_fractional=0.25)
print('QBER before / after onset:', seq.block_qbers[:30].mean().round(4), seq.block_qbers[30:].mean().round(4))

## (b) Circuit-level BB84 vs the NumPy simulator
A few hundred rounds with Z/X preparation, measurement, and an intercept-resend Eve whose measurement is a real circuit execution.

In [ ]:
from qpiai_extras.bb84_circuit import compare_with_numpy_simulator
for k, v in compare_with_numpy_simulator(rounds=400, seed=0).items():
    c = v['circuit']
    print(f"{k}: circuit QBER {c['qber']:.4f} (theory {c['theory']:.4f} +/- {c['std_error']:.4f}, {c['backend']}), NumPy {v['numpy_simulator_qber']:.4f}")

## (c) Figures for the report

In [ ]:
from qpiai_extras.figures import bloch_sphere_figure, circuit_figure
from IPython.display import Image, display
for p in (bloch_sphere_figure(ROOT / 'results/figures/fig_bloch_bb84.png'), circuit_figure(ROOT / 'results/figures/fig_bb84_ir_circuit.png')):
    display(Image(filename=str(p)))